In [48]:
import pandas as pd
import psycopg2
import os

from dotenv import load_dotenv

load_dotenv("../.env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [49]:
conn = psycopg2.connect(
    host="awesome-hw.sdsc.edu",
    port=5432,
    dbname="nourish",
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

def query_db(query):
    """Query the nourish database and return data as a `pd.Dataframe`"""
    try:
        with conn.cursor() as cursor:
            cursor.execute(query)
            columns = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()

        return pd.DataFrame(rows, columns=columns)
    except Exception as e:
        conn.rollback()
        raise e

query_db("SELECT 1")
print(f"Connection successful")

Connection successful


In [50]:
def save_df_to_json(df: pd.DataFrame, filename: str):
    """Save a pandas DataFrame to a JSON file."""
    with open(f"../data/nodes/{filename}", "w") as f:
        df.to_json(f, index=False, orient="records", indent=2)

    print(f"Data saved to data/nodes/{filename}")

##### Entity 1: `State`

In [51]:
query = """
    SELECT
    1 as id,
    'CA' as code,
    'California' as name
"""

state_df = query_db(query)
save_df_to_json(state_df, "state.json")
print(f"Rows: {state_df.shape[0]}, Columns: {state_df.shape[1]}")
state_df.head()

Data saved to data/nodes/state.json
Rows: 1, Columns: 3


,id,code,name
0,1,CA,California


##### Entity 2: `County`

In [52]:
query= """
SELECT id, county as name
FROM county_neighborhoods
"""

county_df = query_db(query)
save_df_to_json(county_df, "county.json")
print(f"Rows: {county_df.shape[0]}, Columns: {county_df.shape[1]}")
county_df.head()

Data saved to data/nodes/county.json
Rows: 58, Columns: 2


,id,name
0,1,Alameda
1,2,Alpine
2,3,Amador
3,4,Butte
4,5,Calaveras


##### Entity 3: `City`

In [53]:
query = """
WITH cte AS (
    SELECT
        c.id,
        c.city,
        ST_Union(op.way) AS geom,  -- merge all polygons for that city
        MIN(ons.osm_id) AS osm_id  -- arbitrary representative ID
    FROM
        city_neighborhoods c
    LEFT JOIN osm_planet_socal_2025.osn_names ons
        ON c.city = ons.name
        AND ons.geom_type = 'polygon'
    LEFT JOIN osm_planet_socal_2025.planet_osm_polygon op
        ON ons.osm_id = op.osm_id
        AND ons.geom_type = 'polygon'
    WHERE
        op.osm_id < 0
        AND county = 'San Diego'
    GROUP BY
        c.city,
        c.id
)
SELECT
    id,
    city as name,
    ST_Transform(geom, 4326) AS geom,
    --st_aswkt(ST_Transform(geom, 4326)) AS geom_wkt,
    NULL AS geom_wkt,
    ST_Centroid(ST_Transform(geom, 4326)) AS centroid

FROM cte; 
"""


city_df = query_db(query)
save_df_to_json(city_df, "city.json")
print(f"Rows: {city_df.shape[0]}, Columns: {city_df.shape[1]}")
city_df.head()

Data saved to data/nodes/city.json
Rows: 52, Columns: 5


,id,name,geom,geom_wkt,centroid
0,8,Carlsbad,0103000020E6100000010000001D0300000EC40D53365A...,None,0101000020E610000049A785B9EB535DC041F8FAFE4F8F...
1,9,Chula Vista,0103000020E610000002000000170600009710BDD6EF47...,None,0101000020E61000003AB1DC6FEC405DC0D69967816650...
2,10,Coronado,0103000020E6100000020000007A0100000DF3D4D97F4E...,None,0101000020E61000000A7445D09F4A5DC054781F833252...
3,11,Del Mar,0103000020E610000001000000CD000000C7EDE1DC7051...,None,0101000020E610000023007032CF505DC09BE5A83B4C7B...
4,12,El Cajon,0103000020E61000000100000013090000DE2230D6B740...,None,0101000020E61000002CD85DA9783D5DC01DEE5A859D66...


##### Entity 4: `Community`

In [54]:
query = """
SELECT id, community as name
FROM community_neighborhoods
WHERE county = 'San Diego';
"""

community_df = query_db(query)
save_df_to_json(community_df, "community.json")
print(f"Rows: {community_df.shape[0]}, Columns: {community_df.shape[1]}")
community_df.head()

Data saved to data/nodes/community.json
Rows: 229, Columns: 2


,id,name
0,60,Midtown
1,55,Linda Vista
2,284,Eastlake Trails
3,285,Eastlake Vistas
4,283,Eastlake Land Swap


##### Entity 5: `Zipcode`

In [55]:
query = """
WITH san_diego_zipcodes AS (

SELECT DISTINCT CAST(unnest(zipcodes) AS TEXT) as zipcode
FROM city_neighborhoods
WHERE county = 'San Diego'

UNION

SELECT DISTINCT CAST(unnest(zipcodes) AS TEXT) as zipcode
FROM community_neighborhoods
WHERE county = 'San Diego'
)

SELECT

t.zipcode,
ST_Transform(s.geom, 4326) AS geom,
ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
ST_Centroid(ST_Transform(geom, 4326)) AS centroid

FROM
san_diego_zipcodes t
JOIN
test_zipcodes s ON t.zipcode = s.zip::text;
"""


zipcode_df = query_db(query)
zipcode_df["id"] = zipcode_df.index + 1
zipcode_df = zipcode_df[["id"]+[col for col in zipcode_df.columns if col != "id"]]
save_df_to_json(zipcode_df, "zipcode.json")
print(f"Rows: {zipcode_df.shape[0]}, Columns: {zipcode_df.shape[1]}")
zipcode_df.head()

Data saved to data/nodes/zipcode.json
Rows: 105, Columns: 5


,id,zipcode,geom,geom_wkt,centroid
0,1,91901,0106000020E61000000100000001030000000400000087...,MULTIPOLYGON(((6417254.00001338 1845596.000065...,0101000020E61000002744E82C37815841BC6005E39C9A...
1,2,91902,0106000020E610000001000000010300000007000000AA...,MULTIPOLYGON(((6324251.97571597 1831249.745293...,0101000020E61000005C8B6E39882258412990A28F05DA...
2,3,91905,0106000020E610000002000000010300000005000000F1...,MULTIPOLYGON(((6528173.3250903 1798504.4708891...,0101000020E6100000AFCF36D40DF758419F1E107E6914...
3,4,91906,0106000020E610000007000000010300000001000000E8...,MULTIPOLYGON(((6528506.99994789 1798956.999839...,0101000020E6100000DF44FF495EC35841D3BC20CCE6C9...
4,5,91910,0106000020E6100000020000000103000000010000000E...,MULTIPOLYGON(((6320499.99535905 1807998.290156...,0101000020E610000042238B4AA51258411E64444D2EA7...


##### Entity 6: `BusinessLocation`

In [56]:
query = """
SELECT    
    id,
    name,
    url,
    address,
    city,
    zip,
    latitude,
    longitude,
    blockgroup,
    categories,
    avg_rating,
    franchise,
    confidence,
    reasoning,
    ST_Transform(geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt
FROM ca_businesses_with_ai_franchise
"""

business_location_df = query_db(query)
business_location_df[business_location_df["zip"].isin(zipcode_df["zipcode"].values)]
save_df_to_json(business_location_df, "business_location.json")
print(f"Rows: {business_location_df.shape[0]}, Columns: {business_location_df.shape[1]}")
business_location_df.head()

Data saved to data/nodes/business_location.json
Rows: 39593, Columns: 16


,id,name,url,address,city,zip,latitude,longitude,blockgroup,categories,avg_rating,franchise,confidence,reasoning,geom,geom_wkt
0,5,Internet Solutions For Less,https://www.google.com/maps/place//data=!4m2!3...,"Internet Solutions For Less, 733 Las Palmas Dr...",Vista,92081,33.1863539,-117.25029649999999,197021,"[Website designer, Design agency, Internet mar...",5,INDEPENDENT,0.85,The business name 'Internet Solutions For Less...,0101000020E610000068739CDB04505DC0B4FDD071DA97...,POINT(-117.2502965 33.1863539)
1,24,Wetzel's Pretzels,https://www.google.com/maps/place//data=!4m2!3...,"Wetzel's Pretzels, 869 W. Harbor Drive, #C2-F,...",San Diego,92101,32.708565199999995,-117.17027739999999,54023,[Pretzel store],3.9,FRANCHISE,0.95,Wetzel's Pretzels is a well-known chain specia...,0101000020E6100000DD0F2ED3E54A5DC0B68AB443B25A...,POINT(-117.1702774 32.7085652)
2,48,Del Mar Golf Center - Pelly's Mini Golf,https://www.google.com/maps/place//data=!4m2!3...,"Del Mar Golf Center - Pelly's Mini Golf, 15555...",Del Mar,92014,32.976022799999996,-117.2534354,83241,"[Golf driving range, Golf instructor, Miniatur...",4.4,INDEPENDENT,0.85,The business name 'Del Mar Golf Center - Pelly...,0101000020E610000084A91C4938505DC03E13AB50EE7C...,POINT(-117.2534354 32.9760228)
3,49,Brainy Actz Escape Rooms San Diego,https://www.google.com/maps/place//data=!4m2!3...,"Brainy Actz Escape Rooms San Diego, 10211 Paci...",San Diego,92121,32.9033799,-117.19029889999999,83462,"[Escape room center, Children's amusement cent...",4.3,FRANCHISE,0.85,The name 'Brainy Actz Escape Rooms' suggests a...,0101000020E6100000BEFD6FDB2D4C5DC08F2EDBF3A173...,POINT(-117.1902989 32.9033799)
4,50,Einstein Bros. Bagels,https://www.google.com/maps/place//data=!4m2!3...,"Einstein Bros. Bagels, 911 Lomas Santa Fe Dr, ...",Solana Beach,92075,32.9942243,-117.25523469999999,173061,"[Bagel shop, Bakery, Breakfast restaurant, Caf...",3.9,FRANCHISE,0.95,Einstein Bros. Bagels is a well-known chain wi...,0101000020E61000007A53ECC355505DC0BAB1EABD427F...,POINT(-117.2552347 32.9942243)


##### Entity 7: `Business`

In [57]:
business_df = business_location_df[["name"]].drop_duplicates().reset_index(drop=True)
business_df["id"] = business_df.index + 1
business_df = business_df[["id"]+[col for col in business_df.columns if col != "id"]]
save_df_to_json(business_df, "business.json")
print(f"Rows: {business_df.shape[0]}, Columns: {business_df.shape[1]}")
business_df.head()

Data saved to data/nodes/business.json
Rows: 32010, Columns: 2


,id,name
0,1,Internet Solutions For Less
1,2,Wetzel's Pretzels
2,3,Del Mar Golf Center - Pelly's Mini Golf
3,4,Brainy Actz Escape Rooms San Diego
4,5,Einstein Bros. Bagels


##### Entity 8: `BlockGroup`

In [58]:
query = """
SELECT
    sbg.ctblockgroup as id,
    bd.std_geography_id AS geo_id,
    sbg.ctblockgroup,
    cs.X1001FY_X,
    S23_EMP,
    N01_BUS,
    cs.X1024_X,
    S22_BUS,
    N14_BUS,
    N37_SALES,
    N35_BUS,
    S16_SALES,
    MEDHINC_CY,
    AVGHINC_CY,
    GINI_FY,
    INDMANU_CY,
    TOTPOP_CY,
    FEM25,
    FEM30,
    FEM35,
    MALE25,
    MALE30,
    MALE35,
    CRMCYTOTC,
    DI100_CY,
    DI150_CY,
    countyfp,
    tractce,
    population,
    apportionm,
    blkgrpce,
    sbg.ogc_fid,
    statefp,
    aggregatio,
    source_cou,
    ST_Transform(sbg.geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
    ST_Centroid(ST_Transform(geom, 4326)) AS centroid
FROM sandag_layer_census_block_groups sbg
LEFT JOIN bgs_sd_imp imp
    ON CAST(sbg.ctblockgroup AS TEXT) = CAST(CONCAT(LTRIM(imp.tractce, '0'), imp.blkgrpce) AS TEXT)
LEFT JOIN esri_business_data bd
    ON TRIM(LEADING '0' FROM SUBSTR(CAST(bd.std_geography_id AS TEXT), 5)) = CAST(sbg.ctblockgroup AS TEXT)
LEFT JOIN esri_consumer_spending_cols cs
    ON TRIM(LEADING '0' FROM SUBSTR(CAST(cs.std_geography_id AS TEXT), 5)) = CAST(sbg.ctblockgroup AS TEXT)
ORDER BY sbg.ctblockgroup ASC;
"""

block_group_df = query_db(query)
save_df_to_json(block_group_df, "block_group.json")
print(f"Rows: {block_group_df.shape[0]}, Columns: {block_group_df.shape[1]}")
block_group_df.head()

Data saved to data/nodes/block_group.json
Rows: 2085, Columns: 38


,id,geo_id,ctblockgroup,x1001fy_x,s23_emp,n01_bus,x1024_x,s22_bus,n14_bus,n37_sales,...,population,apportionm,blkgrpce,ogc_fid,statefp,aggregatio,source_cou,geom,geom_wkt,centroid
0,1001,60730001001,1001,115362302,91,30,349221,0,0,169,...,2.191,2.576,1,1,06,BlockApportionment:US.BlockGroups;PointsLayer:...,USA,0106000020E610000001000000010300000001000000B8...,MULTIPOLYGON(((-117.188567830914 32.7591470904...,0101000020E6100000F9DD3870B24B5DC0799CEE388760...
1,1002,60730001002,1002,179816406,195,59,569578,2,0,12628,...,2.191,2.576,2,2,06,BlockApportionment:US.BlockGroups;PointsLayer:...,USA,0106000020E61000000100000001030000000100000031...,MULTIPOLYGON(((-117.187773001776 32.7572359994...,0101000020E6100000AFC67FB20C4C5DC02FD966214060...
2,2011,60730002011,2011,64440225,345,82,209780,4,0,1080,...,2.191,2.576,1,3,06,BlockApportionment:US.BlockGroups;PointsLayer:...,USA,0106000020E61000000100000001030000000100000088...,MULTIPOLYGON(((-117.169610001679 32.7578239986...,0101000020E61000003299120A344B5DC0EC97B3AF9460...
3,2012,60730002012,2012,95423500,333,154,323824,18,1,3819,...,2.191,2.576,2,4,06,BlockApportionment:US.BlockGroups;PointsLayer:...,USA,0106000020E61000000100000001030000000100000062...,MULTIPOLYGON(((-117.172342000794 32.7556769991...,0101000020E6100000779362E9F44A5DC00471F9E05B60...
4,2021,60730002021,2021,90161948,100,64,259785,4,0,14428,...,2.191,2.576,1,5,06,BlockApportionment:US.BlockGroups;PointsLayer:...,USA,0106000020E6100000010000000103000000010000008D...,MULTIPOLYGON(((-117.172285001534 32.7489329987...,0101000020E610000037A66881594B5DC022ECE625385F...


##### Entity 9: `Zone Location`

In [59]:
query = """
SELECT
id,
zone_name,
imp_date,
ordnum,
shape_length,
shape_area,
legend,
ST_Transform(geom, 4326) AS geom,
ST_AsText(ST_Transform(geom, 4326)) AS geom_ewkt,
ST_Centroid(ST_Transform(geom, 4326)) AS centroid

FROM sandag_layer_zoning_base_sd_new
"""

zone_location_df = query_db(query)
save_df_to_json(zone_location_df, "zone_location.json")
print(f"Rows: {zone_location_df.shape[0]}, Columns: {zone_location_df.shape[1]}")
zone_location_df.head()

Data saved to data/nodes/zone_location.json
Rows: 3677, Columns: 10


,id,zone_name,imp_date,ordnum,shape_length,shape_area,legend,geom,geom_ewkt,centroid
0,1,AG-1-1,1141084800000,R-301263,281.423966,2.601150e+03,"Agricultural-General zone, use package 1, deve...",0106000020E6100000010000000103000000010000000E...,MULTIPOLYGON(((-117.12982648288 33.04642430605...,0101000020E610000058C500454C485DC0EA352115F485...
1,6,AG-1-1,1141084800000,R-301263,3197.909381,2.041669e+05,"Agricultural-General zone, use package 1, deve...",0106000020E61000000100000001030000000100000085...,MULTIPOLYGON(((-117.037822960772 33.0743520276...,0101000020E61000002A608A6D56425DC0F1B74447B289...
2,7,AG-1-1,1141084800000,R-301263,85544.431621,6.369870e+07,"Agricultural-General zone, use package 1, deve...",0106000020E61000000100000001030000000100000040...,MULTIPOLYGON(((-117.106843080255 33.0731224473...,0101000020E610000008FC9AE8AC465DC0D497FCB2CE87...
3,8,AG-1-1,1141084800000,R-301263,17855.440197,7.212937e+06,"Agricultural-General zone, use package 1, deve...",0106000020E610000001000000010300000001000000C8...,MULTIPOLYGON(((-116.972290881461 33.0752706838...,0101000020E6100000312BACEBAC3D5DC09146E002718A...
4,9,AG-1-1,1141084800000,R-301263,168.046707,1.087063e+03,"Agricultural-General zone, use package 1, deve...",0106000020E6100000010000000103000000010000000B...,MULTIPOLYGON(((-116.905763399499 33.0866176989...,0101000020E6100000EA51C073F9395DC06CA3C469168B...


##### Entity 10: `Zone Type`

In [61]:
query = """
SELECT
DISTINCT(zone_name) as name,
legend
FROM sandag_layer_zoning_base_sd_new
"""

zone_type_df = query_db(query)
zone_type_df["id"] = zone_type_df.index + 1
zone_type_df = zone_type_df[["id"]+[col for col in zone_type_df.columns if col != "id"]]
save_df_to_json(zone_type_df, "zone_type.json")
print(f"Rows: {zone_type_df.shape[0]}, Columns: {zone_type_df.shape[1]}")
zone_type_df.head()

Data saved to data/nodes/zone_type.json
Rows: 184, Columns: 3


,id,name,legend
0,1,OF-1-1,Open Space-Floodplain zone (floodplain areas)
1,2,CC-2-5,Other or Undefined Zone
2,3,CC-5-1,Commercial-Community zones (special areas)
3,4,CUPD-CT-2-3,Central Urbanized Planned District
4,5,LJPD-6A,La Jolla Planned District
